# 19 — Window Functions: Rolling, Expanding & Rank

Window functions compute metrics over a sliding or expanding window — critical for
time series analysis, moving averages, cumulative stats, and interview challenges.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Daily sales time series
n_days = 365
daily = pd.DataFrame({
    'date':    pd.date_range('2023-01-01', periods=n_days, freq='D'),
    'revenue': np.random.randint(50000, 500000, n_days) +
               np.sin(np.linspace(0, 4*np.pi, n_days)) * 50000,  # seasonal component
    'orders':  np.random.randint(50, 500, n_days),
    'returns': np.random.randint(1, 30, n_days)
})
daily['revenue'] = daily['revenue'].round(0).astype(int)

# Employee performance dataset
n_emp = 200
employees = pd.DataFrame({
    'emp_id':     range(1001, 1001 + n_emp),
    'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing'], n_emp),
    'salary':     np.random.randint(40000, 180000, n_emp),
    'rating':     np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n_emp, p=[0.05, 0.1, 0.25, 0.4, 0.2]),
    'experience': np.random.randint(1, 20, n_emp)
})

print(f"Daily: {daily.shape}  |  Employees: {employees.shape}")

## 1. rolling() — Sliding Window

In [ ]:
# 7-day rolling average
daily['revenue_7d_avg']    = daily['revenue'].rolling(window=7).mean().round(0)
daily['revenue_30d_avg']   = daily['revenue'].rolling(window=30).mean().round(0)
daily['revenue_7d_std']    = daily['revenue'].rolling(window=7).std().round(0)

# First 6 rows are NaN (not enough history for window=7)
print(daily[['date', 'revenue', 'revenue_7d_avg', 'revenue_30d_avg']].head(10))

In [ ]:
# min_periods: compute even with incomplete windows
daily['revenue_7d_min2'] = daily['revenue'].rolling(window=7, min_periods=2).mean().round(0)
print(daily[['date', 'revenue', 'revenue_7d_avg', 'revenue_7d_min2']].head(8))

In [ ]:
# Multiple rolling aggregations in one call
rolling_stats = daily['revenue'].rolling(window=7).agg(['mean', 'min', 'max', 'std']).round(0)
print(rolling_stats.head(10))

In [ ]:
# Time-based rolling (offset alias) — requires DatetimeIndex
ts = daily.set_index('date')

# 7-day window based on time, not count of rows
ts['revenue_7d_time'] = ts['revenue'].rolling('7D').mean().round(0)
print(ts[['revenue', 'revenue_7d_time']].head(10))

## 2. rolling().apply() — Custom Window Function

In [ ]:
# Apply any function over the rolling window
def revenue_trend(arr):
    """Returns 1 if last value > first value, else -1."""
    return 1 if arr[-1] > arr[0] else -1

daily['trend_7d'] = daily['revenue'].rolling(7).apply(revenue_trend, raw=True)
print("Trend direction (7d):")
print(daily['trend_7d'].value_counts())

## 3. expanding() — Cumulative Window

In [ ]:
# expanding(): window grows from start to current row
daily['cumulative_revenue'] = daily['revenue'].expanding().sum()
daily['ytd_avg_revenue']    = daily['revenue'].expanding().mean().round(0)
daily['all_time_max']       = daily['revenue'].expanding().max()

print(daily[['date', 'revenue', 'cumulative_revenue', 'ytd_avg_revenue', 'all_time_max']].head(10))

In [ ]:
# running total vs cumsum
print("cumsum():")
print(daily['orders'].cumsum().head(5).tolist())

print("expanding().sum():")
print(daily['orders'].expanding().sum().head(5).tolist())
# Both are identical! cumsum() is faster.

## 4. Exponential Weighted Moving Average (EWMA)

In [ ]:
# ewm(): exponentially weighted — more recent data gets higher weight
daily['revenue_ewm_7']  = daily['revenue'].ewm(span=7,  adjust=False).mean().round(0)
daily['revenue_ewm_30'] = daily['revenue'].ewm(span=30, adjust=False).mean().round(0)

print("Regular vs EWM averages:")
print(daily[['date', 'revenue', 'revenue_7d_avg', 'revenue_ewm_7']].head(15))

In [ ]:
# EWM has no NaN issue — starts computing from row 0
print(f"NaN in rolling 7d:  {daily['revenue_7d_avg'].isna().sum()}")
print(f"NaN in ewm span=7:  {daily['revenue_ewm_7'].isna().sum()}")

## 5. rank() — Ranking Within Groups

In [ ]:
# rank() methods — handling ties
s = pd.Series([85000, 90000, 85000, 110000, 70000])

print("Values:",    s.tolist())
print("average:",   s.rank(method='average').tolist())   # default
print("min:",       s.rank(method='min').tolist())        # lower rank to ties
print("max:",       s.rank(method='max').tolist())        # higher rank to ties
print("dense:",     s.rank(method='dense').tolist())      # no gaps in ranking
print("first:",     s.rank(method='first').tolist())      # by order of appearance

In [ ]:
# Rank salary descending — highest salary = rank 1
employees['salary_rank'] = employees['salary'].rank(ascending=False, method='dense').astype(int)

# Rank within department
employees['dept_salary_rank'] = (
    employees.groupby('department')['salary']
    .rank(ascending=False, method='dense')
    .astype(int)
)

print(employees[['emp_id', 'department', 'salary', 'salary_rank', 'dept_salary_rank']].head(10))

In [ ]:
# Percentile rank — pct=True
employees['salary_percentile'] = (employees['salary'].rank(pct=True) * 100).round(1)
print("Top 10% earners:")
print(
    employees[employees['salary_percentile'] >= 90]
    [['emp_id', 'department', 'salary', 'salary_percentile']]
    .sort_values('salary', ascending=False)
    .head(10)
)

## 6. nlargest() / nsmallest() — Top/Bottom N

In [ ]:
# Top 5 revenue days
print("Top 5 revenue days:")
print(daily.nlargest(5, 'revenue')[['date', 'revenue', 'orders']])

# Bottom 5 revenue days
print("\nBottom 5 revenue days:")
print(daily.nsmallest(5, 'revenue')[['date', 'revenue', 'orders']])

## 7. shift() — Lag Features for ML

In [ ]:
# shift() creates lag (or lead) features — essential for ML pipelines
daily['revenue_lag1']  = daily['revenue'].shift(1)    # yesterday
daily['revenue_lag7']  = daily['revenue'].shift(7)    # same day last week
daily['revenue_lead1'] = daily['revenue'].shift(-1)   # tomorrow (for target creation)

# Day-over-day and week-over-week change
daily['revenue_dod']   = daily['revenue'] - daily['revenue_lag1']  # absolute change
daily['revenue_dod_pct'] = (daily['revenue_dod'] / daily['revenue_lag1'] * 100).round(2)
daily['revenue_wow_pct'] = ((daily['revenue'] - daily['revenue_lag7']) / daily['revenue_lag7'] * 100).round(2)

print(daily[['date', 'revenue', 'revenue_lag1', 'revenue_dod_pct', 'revenue_wow_pct']].head(12))

## 8. diff() — Simple Differencing

In [ ]:
# diff() = shift(1) shortcut for period-over-period difference
daily['revenue_diff1'] = daily['revenue'].diff(1)   # day-over-day
daily['revenue_diff7'] = daily['revenue'].diff(7)   # week-over-week

print(daily[['date', 'revenue', 'revenue_diff1', 'revenue_diff7']].head(10))

## 9. Production — Feature Engineering Pipeline

In [ ]:
def create_time_features(df: pd.DataFrame, target_col: str = 'revenue') -> pd.DataFrame:
    """Create time-series features for ML."""
    df = df.copy()
    
    # Lags
    for lag in [1, 7, 14, 30]:
        df[f'{target_col}_lag{lag}'] = df[target_col].shift(lag)
    
    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'{target_col}_rolling{window}_mean'] = df[target_col].rolling(window).mean().round(0)
        df[f'{target_col}_rolling{window}_std']  = df[target_col].rolling(window).std().round(0)
    
    # Percent changes
    df[f'{target_col}_pct_change_1d']  = df[target_col].pct_change(1).round(4)
    df[f'{target_col}_pct_change_7d']  = df[target_col].pct_change(7).round(4)
    
    # Expanding
    df[f'{target_col}_cum_sum']  = df[target_col].cumsum()
    df[f'{target_col}_cum_max']  = df[target_col].cummax()
    
    # EWM
    df[f'{target_col}_ewm7'] = df[target_col].ewm(span=7, adjust=False).mean().round(0)
    
    return df

features = create_time_features(daily[['date', 'revenue', 'orders']].head(60))
print(f"Feature columns: {features.shape[1]}")
print(features.columns.tolist())

## 10. Rolling GroupBy — Rolling Within Groups

In [ ]:
# Per-department rolling average salary
# Generate time-series employee data
emp_monthly = pd.DataFrame({
    'month':      np.repeat(pd.date_range('2023-01', periods=12, freq='ME'), 4),
    'department': ['Engineering', 'Sales', 'HR', 'Marketing'] * 12,
    'avg_salary': np.random.randint(60000, 120000, 48)
})

emp_monthly['rolling_3m_salary'] = (
    emp_monthly.sort_values(['department', 'month'])
    .groupby('department')['avg_salary']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
    .round(0)
)

print(emp_monthly.sort_values(['department', 'month']).head(15))